In [1]:
import scipy
import numpy as np
import pandas as pd

import torch
import torchaudio
import torchaudio.transforms as T
import sys
import torch.optim as optim
from torch.optim.lr_scheduler import CyclicLR
from speechbrain.inference.speaker import EncoderClassifier
from transformers import AutoFeatureExtractor, AutoModel

sys.path.append('/om2/user/salavill/misc/voice-speech-metamers/')
from utils import *
from learner import Learner
from tokenizer import Tokenizer
from decoder import Speech_Decoder_Linear, Speaker_Decoder_Linear
from encoder import Speaker_Encoder, Speech_Encoder, Joint_Encoder

In [2]:
sr = 16000

# Load in ECAPA - Voice model

In [9]:
ecapa_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
# target = model.encode_batch(signal)[0]

# Load in Whisper - Speech Model

In [4]:
# load in model 
whisper_feature_extractor = AutoFeatureExtractor.from_pretrained("openai/whisper-base")
whisper_encoder = AutoModel.from_pretrained("openai/whisper-base")#, cache_dir=cache_dir)
decoder_input_ids = torch.tensor([[1, 1]]) * whisper_encoder.config.decoder_start_token_id
whisper_encoder.eval()

def run_whisper(input, noise=False):
    """
    runs the whisper model when given audio input
    """
    input = whisper_feature_extractor(input.detach().cpu(), sampling_rate=sr, return_tensors="pt").input_features
    if noise:
        input = input.clone().requires_grad_()
    output = whisper_encoder(input, decoder_input_ids=decoder_input_ids)
    return output.encoder_last_hidden_state.mean(1)

# target = run_model(signal)


# Load in Joint Model

In [17]:
# load in model 
config_path = "../config.yaml"

# Load config file
config = load_yaml_config(config_path)

#define a tokenizer for the vocabulary
tokenizer = Tokenizer(**config.text)

speaker_encoder = Speaker_Encoder(config.encoder.model_cache)
speech_encoder = Speech_Encoder(config.encoder.model_cache)

#define joint encoder
saganet = Joint_Encoder(config.saganet.d_model,
                        config.saganet.num_head,
                        config.saganet.dim_feedforward,
                        config.saganet.num_layers)

#define decoders
speech_decoder = Speech_Decoder_Linear()
speaker_decoder = Speaker_Decoder_Linear()


checkpoint = "/om2/user/annesyab/SLP_Project_2024/saganet/saganet_d-704_atthead-4/best15-val_loss2.35.ckpt"
saganet = Learner.load_from_checkpoint(checkpoint_path=checkpoint,
                                                config=config, 
                                                tokenizer=tokenizer,
                                                speech_encoder=speech_encoder,
                                                speaker_encoder=speaker_encoder,
                                                joint_encoder=saganet,
                                                speech_decoder=speech_decoder,
                                                speaker_decoder = speaker_decoder,)

print('Loaded in joint model')

# Get target embedding by running signal through model
# target, _ = model(signal)

In [12]:
saganet()

NameError: name 'saganet' is not defined

# Create Joint Babble Data

In [ ]:
def combine_signal_and_noise(signal, noise, snr, mean_subtract=True):
    '''
    Adds noise to signal with the specified signal-to-noise ratio (snr).
    If snr is finite, the noise waveform is rescaled and added to the
    signal waveform. If snr is positive infinity, returned waveform is
    equal to the signal waveform. If snr is negative inifinity, returned
    waveform is equal to the noise waveform.
    
    Args
    ----
    signal (np.ndarray): signal waveform
    noise (np.ndarray): noise waveform
    snr (float): signal-to-noise ratio in dB
    mean_subtract (bool): if True, signal and noise are first de-meaned
        (mean_subtract=True is important for accurate snr computation)
    
    Returns
    -------
    signal_and_noise (np.ndarray) signal in noise waveform
    '''
    rms = lambda stim: np.sqrt(np.mean(stim * stim))

    if mean_subtract:
        signal = signal - np.mean(signal)
        noise = noise - np.mean(noise)        
    if np.isinf(snr) and snr > 0:
        signal_and_noise = signal
    elif np.isinf(snr) and snr < 0:
        signal_and_noise = noise
    else:
        rms_noise_scaling = rms(signal) / (rms(noise) * np.power(10, snr / 20))
        signal_and_noise = signal + rms_noise_scaling * noise
    return signal_and_noise

In [ ]:
import glob
backgrounds = glob.glob('/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/background_cv08talkerbabble/*.wav')

NameError: name 'run_whisper' is not defined

In [ ]:
pd.read_csv('/om2/user/gelbanna/commonvoice_data_curated.csv')

,Unnamed: 0,client_id,sr,wav_path,total_file_duration_in_s,gender,sentence,speaker_int,split
0,1480475,372293e65cdab88771e028a4351651ab2eff64438ddafc...,48000,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,9.192,male,both accounts concur that green first heard jo...,0,train
1,1487979,372293e65cdab88771e028a4351651ab2eff64438ddafc...,48000,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,4.656,male,everything was deadly still,0,train
2,1478405,372293e65cdab88771e028a4351651ab2eff64438ddafc...,48000,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,7.200,male,in spite of this blackthorne becomes a trusted...,0,train
3,1491837,372293e65cdab88771e028a4351651ab2eff64438ddafc...,48000,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,4.752,male,how foolish to reveal those talons to him,0,train
4,1491882,372293e65cdab88771e028a4351651ab2eff64438ddafc...,48000,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,4.584,male,the two knives were left implanted,0,train
...,...,...,...,...,...,...,...,...,...
78431,984682,4cbced96a01d967939d63d4d35b3068a70aabb0dcc567b...,32000,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,4.248,female,and then a shadow came between rocco and the sun,199,test
78432,984686,4cbced96a01d967939d63d4d35b3068a70aabb0dcc567b...,32000,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,4.320,female,contemporary reviews were generally favorable,199,test
78433,984693,4cbced96a01d967939d63d4d35b3068a70aabb0dcc567b...,32000,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,4.356,female,the station acquired the nickname of hooligan ...,199,test
78434,984697,4cbced96a01d967939d63d4d35b3068a70aabb0dcc567b...,32000,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,5.688,female,numerous alumni are also involved in postcolle...,199,test


In [7]:
sig_path = pd.read_csv('/om2/user/gelbanna/commonvoice_data_curated.csv').query('split == "test" and total_file_duration_in_s > 2').iloc[0].wav_path
noise_path = backgrounds[0]
sr = 16000

NameError: name 'run_whisper' is not defined

In [24]:

signal, fs = torchaudio.load(sig_path)
if fs != sr:
    # make sure to resample to appropriate frequency
    print('resampling audio')
    resampler = T.Resample(fs, sr, dtype=signal.dtype)
    signal = resampler(signal)

if len(signal.shape)>1:
    # Reshape signal as necessary
    signal = torch.squeeze(signal)

signal = signal.numpy()

resampling audio


In [37]:
noise.shape, signal.shape

((48000,), (115200,))

In [36]:
noise, fs = torchaudio.load(noise_path)
if fs != sr:
    # make sure to resample to appropriate frequency
    print('resampling audio')
    resampler = T.Resample(fs, sr, dtype=noise.dtype)
    noise = resampler(noise)

if len(noise.shape)>1:
    # Reshape signal as necessary
    noise = torch.squeeze(noise)

noise = noise.numpy()

signal = combine_signal_and_noise(signal, noise, 25, mean_subtract=True)

resampling audio


ValueError: operands could not be broadcast together with shapes (115200,) (48000,) 

In [28]:
run_whisper(torch.from_numpy(signal))

tensor([[-1.9840e-01, -6.1409e-01,  3.0537e-01, -1.1975e+00,  1.0036e-01,
          1.7028e+00, -4.4624e-01, -9.5249e-03, -9.2293e-01, -7.5952e-02,
         -1.5684e-01,  5.7124e-01, -2.2324e-01,  1.0309e+00,  8.8013e-01,
          3.7764e-04, -1.3836e-02,  3.4503e-01,  8.1493e-01, -9.3862e-04,
          4.7322e-03, -6.9092e-03, -9.7092e-02,  2.2744e-01, -3.0087e-01,
          2.3260e+00, -1.6739e-01,  8.3928e-01, -5.8203e-01, -1.8112e+00,
         -3.5937e-01,  6.1444e-01, -1.1309e-01, -6.2463e-02,  1.5556e-01,
          9.0267e-02,  1.3020e-01, -2.5497e-01,  1.7304e-01,  1.7952e-01,
         -1.2829e-01,  1.6095e+00,  5.5677e-01,  4.4332e-01, -4.0025e-01,
          3.1468e-02, -1.6480e-01, -4.1340e-01,  1.4972e-01,  1.8336e-01,
         -2.8867e-01, -2.9536e-01,  5.7337e-01,  3.4407e-02,  1.9870e-01,
          3.7185e-01,  1.6838e-01, -3.1914e-01,  1.6848e-01, -1.2977e-01,
         -7.2072e-02,  1.6693e+00, -2.5528e-01,  7.0030e-02,  3.5411e-01,
          2.7643e-01,  1.8051e-01, -4.

# Test Models